In [2]:
import os
import duckdb
import pandas as pd

In [3]:
from pathlib import Path
from dotenv import load_dotenv

repo_root = Path.cwd().parents[1]
load_dotenv(repo_root / ".env", override=True)

token = os.environ["HF_TOKEN"]

print("Token loaded:", bool(token))

Token loaded: True


In [4]:
con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')"
)

print("Connected.")

Connected.


In [5]:
rel = "hf://datasets/FlyRank/internship-warehouse"

print(rel)

hf://datasets/FlyRank/internship-warehouse


In [7]:
feature_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-01-01'
  AND report_date < DATE '2026-04-01'
GROUP BY
    client_hash_id,
    content_hash_id
"""

features = con.sql(feature_query).df()

features.head()

,client_hash_id,content_hash_id,avg_impressions,avg_clicks
0,client_3ffa76342f366962,content_31dd2c50b9eab5e9,0.178082,0.0
1,client_3ffa76342f366962,content_7a447c2d3296a09d,0.042254,0.0
2,client_3ffa76342f366962,content_de0c543bb5e8c7f7,0.085714,0.0
3,client_3ffa76342f366962,content_6bc62151e86766f3,0.000000,0.0
4,client_3ffa76342f366962,content_8bc5fa4297f8e126,0.000000,0.0


In [8]:
schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 1
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [9]:
print(
    schema[
        schema["column_name"]
        .str.contains(
            "impression|click|ctr|position|date",
            case=False,
            regex=True
        )
    ][["column_name", "column_type"]]
)

         column_name column_type
0        report_date        DATE
7    gsc_impressions      BIGINT
8         gsc_clicks      BIGINT
9   gsc_sum_position      BIGINT
10  gsc_avg_position      DOUBLE


In [11]:
features = con.sql(feature_query).df()

In [13]:
from pathlib import Path

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "lane2_features.csv"

features.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Rows:", len(features))
print("Columns:", len(features.columns))

Saved: ..\outputs\lane2_features.csv
Rows: 349411
Columns: 4


In [14]:
feature_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks,
    AVG(gsc_avg_position) AS avg_position
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-01-01'
  AND report_date < DATE '2026-04-01'
GROUP BY
    client_hash_id,
    content_hash_id
"""

features = con.sql(feature_query).df()

print("Feature rows:", len(features))
features.head()

Feature rows: 349411


,client_hash_id,content_hash_id,avg_impressions,avg_clicks,avg_position
0,client_3ffa76342f366962,content_b8d71ac9305b779d,0.000000,0.000000,NaN
1,client_3ffa76342f366962,content_630de85aa9a7e467,0.000000,0.000000,NaN
2,client_3ffa76342f366962,content_d5162724c0581f01,0.000000,0.000000,NaN
3,client_3ffa76342f366962,content_5573434837db89c5,3.454545,0.136364,6.935899
4,client_3ffa76342f366962,content_d2dc06e56fcf3f3b,0.000000,0.000000,NaN


In [15]:
label_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS future_avg_impressions
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-04-01'
  AND report_date < DATE '2026-05-01'
GROUP BY
    client_hash_id,
    content_hash_id
"""

future = con.sql(label_query).df()

print("Future rows:", len(future))
future.head()

Future rows: 362172


,client_hash_id,content_hash_id,future_avg_impressions
0,client_2b4306c3ed003f01,content_a822ad92acb47f99,0.0
1,client_2b4306c3ed003f01,content_78973ed32502b788,0.0
2,client_2b4306c3ed003f01,content_26a83e47da126068,0.0
3,client_2b4306c3ed003f01,content_71207875ac119d31,0.0
4,client_2b4306c3ed003f01,content_a64f78e55449ec46,0.0


In [16]:
dataset = features.merge(
    future,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Joined rows:", len(dataset))
dataset.head()

Joined rows: 331436


,client_hash_id,content_hash_id,avg_impressions,avg_clicks,avg_position,future_avg_impressions
0,client_3ffa76342f366962,content_b8d71ac9305b779d,0.000000,0.000000,NaN,0.000000
1,client_3ffa76342f366962,content_630de85aa9a7e467,0.000000,0.000000,NaN,0.000000
2,client_3ffa76342f366962,content_d5162724c0581f01,0.000000,0.000000,NaN,0.000000
3,client_3ffa76342f366962,content_5573434837db89c5,3.454545,0.136364,6.935899,1.766667
4,client_3ffa76342f366962,content_d2dc06e56fcf3f3b,0.000000,0.000000,NaN,0.000000


In [17]:
dataset["future_decline"] = (
    dataset["future_avg_impressions"] < dataset["avg_impressions"] * 0.80
).astype(int)

print(dataset["future_decline"].value_counts())

future_decline
0    232868
1     98568
Name: count, dtype: int64


In [18]:
print(
    dataset[
        [
            "avg_impressions",
            "future_avg_impressions",
            "future_decline"
        ]
    ].head(10)
)

   avg_impressions  future_avg_impressions  future_decline
0         0.000000                0.000000               0
1         0.000000                0.000000               0
2         0.000000                0.000000               0
3         3.454545                1.766667               1
4         0.000000                0.000000               0
5         0.000000                0.000000               0
6         0.000000                0.000000               0
7         0.000000                0.000000               0
8         0.000000                0.000000               0
9         0.014286                0.033333               0


In [19]:
print("Target distribution:")
print(dataset["future_decline"].value_counts())

Target distribution:
future_decline
0    232868
1     98568
Name: count, dtype: int64


In [20]:
dataset = dataset.drop(
    columns=["future_avg_impressions"]
)

print(dataset.columns.tolist())

['client_hash_id', 'content_hash_id', 'avg_impressions', 'avg_clicks', 'avg_position', 'future_decline']


In [21]:
from pathlib import Path

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "lane2_training_data.csv"

dataset.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)
print("Rows:", len(dataset))
print("Columns:", len(dataset.columns))

Saved: ..\outputs\lane2_training_data.csv
Rows: 331436
Columns: 6
